In [1]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 86.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.9 MB/s eta 0:00:0000:01


In [2]:
import os
import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
from huggingface_hub import login


login("") 

In [5]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
DATA_PATH = "/kaggle/input/datasets/priyansh384/hindi-conversational/hindi_1third.jsonl"
OUTPUT_DIR = "/kaggle/working/llama3-instruction-finetuned"

MAX_SAMPLES = 50000   # change or remove if you want full data
MAX_LENGTH = 512
SEED = 42

In [6]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

if MAX_SAMPLES is not None and len(dataset) > MAX_SAMPLES:
    dataset = dataset.select(range(MAX_SAMPLES))

split = dataset.train_test_split(test_size=0.02, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print(train_dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Train: 2669
Eval: 55
['instruction', 'input', 'output']


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    instruction = str(example.get("instruction", "")).strip()
    input_text = str(example.get("input", "")).strip()
    output_text = str(example.get("output", "")).strip()

    user_content = instruction
    if input_text:
        user_content += "\n\n" + input_text

    messages = [
        {"role": "system", "content": "You are a helpful medical assistant. Give safe, clear, and concise medical guidance."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_text},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_example, remove_columns=eval_dataset.column_names)

print(train_dataset[0]["text"][:1000])

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Map:   0%|          | 0/2669 [00:00<?, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful medical assistant. Give safe, clear, and concise medical guidance.<|eot_id|><|start_header_id|>user<|end_header_id|>

एक योग्य चिकित्सा चिकित्सक के रूप में कार्य करें और सुरक्षित चिकित्सा मार्गदर्शन प्रदान करें।

मैं इसे 15 अप्रैल से पहले या 15 अप्रैल को समाप्त करने के लिए तैयार करना चाहता हूं क्योंकि मुझे 16 अप्रैल को यात्रा करनी है। मैंने इस उद्देश्य के लिए एक दवा के रूप में रेजेस्टेरॉन के बारे में सुना है - 3 दिनों के लिए एक दिन में 3 बार खुराक, पीरियड्स 3 दिनों के लिए खुराक के बाद 3 दिनों में शुरू हो जाएगा। Plz पुष्टि करें। 575 Views v<|eot_id|><|start_header_id|>assistant<|end_header_id|>

कोई भी इस बात की पुष्टि नहीं कर सकता है.regesterone के बाद पीरियड्स 3 से 15 दिनों के बीच कहीं भी शुरू हो सकते हैं।<|eot_id|>


In [8]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [9]:


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [10]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    max_length=MAX_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to=[],
    remove_unused_columns=False,
    optim="paged_adamw_8bit",
    bf16=use_bf16,
    fp16=not use_bf16,
    seed=SEED,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
checkpoint_path = "/kaggle/input/models/priyansh384/checkpont/pytorch/default/1/llama3-instruction-finetuned/checkpoint-84"

In [14]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train(resume_from_checkpoint=checkpoint_path)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)

Epoch,Training Loss,Validation Loss
